# Lezione 12 — Pratica: chiudi il tuo caso d'uso NOAA/JFK

Questa è la pratica finale del corso: estenderai la pipeline della lezione con la tua
propria analisi, come faresti in un progetto reale.

## Esercizio 1 — Esplorare una variabile aggiuntiva

**Consegna:** usando il DataFrame `clima` già pulito della lezione, disegna la
tendenza oraria di `HOURLYRelativeHumidity` (umidità relativa) durante l'anno 2017,
e calcola la sua correlazione con `HOURLYDRYBULBTEMPF` (temperatura).

**Suggerimento:** usa `.loc["2017", colonna]` per filtrare l'anno, e `.corr()` tra due
colonne di un DataFrame (o `Series.corr(Series)`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

# TODO: disegna la tendenza di HOURLYRelativeHumidity durante il 2017

# TODO: calcola la correlazione tra HOURLYRelativeHumidity e HOURLYDRYBULBTEMPF
correlacion_humedad_temp = None

print(f"Correlación humedad-temperatura: {correlacion_humedad_temp:.2f}")


<details>
<summary>💡 Vedi soluzione</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

clima.loc["2017", "HOURLYRelativeHumidity"].plot(figsize=(10, 4))
plt.title("Humedad relativa horaria en JFK durante 2017")
plt.ylabel("% de humedad")
plt.show()

correlacion_humedad_temp = clima["HOURLYRelativeHumidity"].corr(clima["HOURLYDRYBULBTEMPF"])
print(f"Correlación humedad-temperatura: {correlacion_humedad_temp:.2f}")
```
</details>

## Esercizio 2 — Puoi superare il modello di riferimento?

**Consegna:** in lezione hai confrontato il modello "senza cambiamento" con la media mobile di
3 ore. Ora prova con una media mobile di **6 ore** e confronta il suo RMSE con
entrambi i modelli precedenti. Quale dei tre funziona meglio?

**Suggerimento:** il codice è identico a quello della media mobile di 3 ore, cambia solo il
numero dentro `.rolling(...)`.

In [ ]:
from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

prediccion_baseline = temperatura.shift(1).dropna()
rmse_baseline = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_baseline.index], prediccion_baseline
))

prediccion_media_3h = temperatura.rolling(3).mean().shift(1).dropna()
rmse_media_3h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_3h.index], prediccion_media_3h
))

# TODO: calcola prediccion_media_6h con .rolling(6).mean().shift(1).dropna()
prediccion_media_6h = None

# TODO: calcola il suo RMSE allo stesso modo dei due modelli precedenti
rmse_media_6h = None

print(f"RMSE sin cambio:      {rmse_baseline:.2f} °F")
print(f"RMSE media móvil 3h:  {rmse_media_3h:.2f} °F")
print(f"RMSE media móvil 6h:  {rmse_media_6h:.2f} °F")


<details>
<summary>💡 Vedi soluzione</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

prediccion_baseline = temperatura.shift(1).dropna()
rmse_baseline = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_baseline.index], prediccion_baseline
))

prediccion_media_3h = temperatura.rolling(3).mean().shift(1).dropna()
rmse_media_3h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_3h.index], prediccion_media_3h
))

prediccion_media_6h = temperatura.rolling(6).mean().shift(1).dropna()
rmse_media_6h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_6h.index], prediccion_media_6h
))

print(f"RMSE sin cambio:      {rmse_baseline:.2f} °F")
print(f"RMSE media móvil 3h:  {rmse_media_3h:.2f} °F")
print(f"RMSE media móvil 6h:  {rmse_media_6h:.2f} °F")
```

Normalmente, finestre più lunghe appiattiscono troppo la previsione e peggiorano l'
RMSE rispetto a finestre corte — è un buon esempio del fatto che "più complesso" non sempre
significa "migliore". Congratulazioni per aver completato il corso!
</details>